In [ ]:

!pip install xgboost

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    brier_score_loss,
    precision_score,
    recall_score,
    f1_score,
    fbeta_score,
    confusion_matrix,
    PrecisionRecallDisplay,
    RocCurveDisplay
)

from sklearn.calibration import calibration_curve

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

from xgboost import XGBClassifier

RANDOM_STATE = 42

# Assignment 5: Bankruptcy Prediction

## 1. Problem Definition

The target variable is `Bankrupt?`.

The positive class is `1`, meaning the company went bankrupt.

This is a binary classification problem. The business goal is to help a risk team identify companies that may go bankrupt.

The classes are imbalanced because bankrupt companies are rare.

Missing a bankrupt company is worse than incorrectly flagging a healthy company.

The primary discrimination metric will be PR-AUC.

The calibration metric will be Brier score.

In [ ]:
df = pd.read_csv("data.csv")

print("Dataset shape:", df.shape)
df.head()

## 2. Quick Exploratory Data Analysis (EDA)

In [ ]:
print("Dataset shape:", df.shape)

print("\nTarget distribution:")
print(df["Bankrupt?"].value_counts())

print("\nTarget distribution percentage:")
print(df["Bankrupt?"].value_counts(normalize=True) * 100)

print("\nTotal missing values:")
print(df.isna().sum().sum())

print("\nDuplicate rows:")
print(df.duplicated().sum())

In [ ]:
sns.countplot(x="Bankrupt?", data=df)
plt.title("Target Distribution")
plt.show()

Observation: The bankrupt class is rare, so accuracy is not a good primary metric. I will use PR-AUC because it focuses more on performance for the positive class.

## 3. Train, Validation, and Test Split

I split the data into 70% training, 15% validation, and 15% test.

I used stratified sampling so that each split keeps a similar bankrupt/non-bankrupt class balance.

The test set will not be used until the final evaluation.

In [ ]:
# Separate features and target
X = df.drop(columns=["Bankrupt?"])
y = df["Bankrupt?"]

# First split: 70% train, 30% temporary
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    stratify=y,
    random_state=RANDOM_STATE
)

# Second split: split temporary data into 15% validation and 15% test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    stratify=y_temp,
    random_state=RANDOM_STATE
)

print("Train shape:", X_train.shape)
print("Validation shape:", X_val.shape)
print("Test shape:", X_test.shape)

In [ ]:
def show_class_balance(y_data, name):
    counts = y_data.value_counts()
    percentages = y_data.value_counts(normalize=True) * 100
    
    balance = pd.DataFrame({
        "count": counts,
        "percentage": percentages.round(2)
    })
    
    print(f"\n{name} class balance:")
    print(balance)

show_class_balance(y_train, "Train")
show_class_balance(y_val, "Validation")
show_class_balance(y_test, "Test")

## 4. Preprocessing

- The target variable has already been separated from the features.
- Most features are numeric.
- I will handle missing values using median imputation.
- Scaling will only be applied for models that need it (e.g., Logistic Regression).
- XGBoost does not require feature scaling.